# Improve rural / other areas (run today)

Four experiments, independent. The shipped model stays **V12 city-trained** unless one of these clearly beats it on the
same held-out test set. Needs a GPU. Run cells 1-3 of the main notebook first (PROJECT, OUT, RUNS, BASE defined).
Each training uses 2 seeds to fit the day; add seed 2 later for the winner. Each writes to its own folder, and the
comparison is always against the shipped V12 on the SAME regions.

In [2]:
# 0. Paths (skip if the main notebook already defined them)
import os, glob, shutil
BASE = globals().get('BASE', '/home/jovyan/vault/lulc'); OUT = globals().get('OUT', f'{BASE}/raw')
RUNS = globals().get('RUNS', f'{BASE}/runs'); PROJECT = globals().get('PROJECT', 'sar-lulc')
os.chdir('/home/jovyan/vault/sar-lulc')
def copy_baselines(dst):
    os.makedirs(dst, exist_ok=True)
    for v in ['V0_scratch_1m','V1_imagenet_1m']:
        if os.path.isdir(f'{RUNS}/{v}') and not os.path.isdir(f'{dst}/{v}'): shutil.copytree(f'{RUNS}/{v}', f'{dst}/{v}')
print('OUT', OUT, '| RUNS', RUNS)

OUT /home/jovyan/vault/lulc/raw | RUNS /home/jovyan/vault/lulc/runs


In [3]:
!pip -q install -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
crewai 1.7.2 requires opentelemetry-api~=1.34.0, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-proto 1.34.1 requires protobuf<6.0,>=5.0, but you have protobuf 6.33.6 which is incompatible.
opentelemetry-sdk 1.34.1 requires opentelemetry-api==1.34.1, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-semantic-conventions 0.55b1 requires opentelemetry-api==1.34.1, but you have opentelemetry-api 1.45.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [4]:
# 3. Earth Engine login (prints a link; open it, sign in, paste the code back here)
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=PROJECT)

/home/jovyan/.local/lib/python3.11/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.74.0. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


Enter verification code:  [redacted]



Successfully saved authorization token.


## 1. Cheap post-processing (no retrain)
Measures cleaner maps, not mIoU. `--prob-smooth` averages the class probabilities over the 3-month window instead of
voting; the confidence mask (in the prototype) hides low-confidence pixels. Run the eval with --prob-smooth and read the
`model+probmean3` false-change row vs `model+majority3`.

In [5]:
# 1. Post-processing comparison on the shipped V12 (writes eval_..._postproc)
%run scripts/evaluate.py --data $OUT --runs $RUNS --variant V12_window3_balanced_stats --prob-smooth --tag _postproc

3 seeds, window 3, coherence False, yearly stats True, year 2021
Esri scheme: original
prague       mIoU vs DW 0.573  OA 0.876
frankfurt    mIoU vs DW 0.588  OA 0.874
milan        mIoU vs DW 0.515  OA 0.886
valencia     mIoU vs DW 0.469  OA 0.791
north_china_plain mIoU vs DW 0.352  OA 0.854
appalachians mIoU vs DW 0.143  OA 0.777
western_desert mIoU vs DW 1.000  OA 1.000
great_basin  mIoU vs DW 0.083  OA 0.400
lake_balaton mIoU vs DW 0.563  OA 0.952
za_freestate_maize mIoU vs DW 0.355  OA 0.704
za_lowveld_savanna mIoU vs DW 0.157  OA 0.224
za_karoo_shrub mIoU vs DW 0.158  OA 0.613
za_capetown  mIoU vs DW 0.482  OA 0.887
india_punjab mIoU vs DW 0.289  OA 0.797
deccan_india mIoU vs DW 0.310  OA 0.585
java_rice    mIoU vs DW 0.207  OA 0.798
mekong_delta mIoU vs DW 0.359  OA 0.726
amazon_para  mIoU vs DW 0.427  OA 0.905
cerrado_brazil mIoU vs DW 0.385  OA 0.661
australia_outback mIoU vs DW 0.204  OA 0.718
altiplano_andes mIoU vs DW 0.345  OA 0.887
sahel_niger  mIoU vs DW 0.039  OA 0.180
io

## 2. Targeted grass/savanna sites (one small retrain)
Adds 4 grass/savanna sites to training (not 12 diverse ones). Export if needed, then edit regions.yaml to make them
train, retrain V12, and compare on the held-out set. Watch Water and Grass/shrub.

In [6]:
# 2a. Export the 4 targeted sites (coverage check first)
GRASS = 'ethiopia_highlands cerrado_brazil kazakh_steppe texas_rangeland'
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 --regions $GRASS
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $GRASS

[1/4] cerrado_brazil 2021 done
[2/4] kazakh_steppe 2021 done
[3/4] ethiopia_highlands 2021 done
[4/4] texas_rangeland 2021 done

Per year, all regions pooled
      s1_median_per_month  s1_min_month  dw_cov1_mean  dw_cov1_winter  dw_cov1_worst_month  dw_cov2_mean  region_months_cov1_below_80pct  region_months
year                                                                                                                                                   
2021                  3.0             1          0.98            0.94                  0.0          0.92                               1             48

Per region and year

cerrado_brazil
      s1_median_per_month  s1_min_month  dw_cov1_mean  dw_cov1_winter  dw_cov1_worst_month
year                                                                                      
2021                  2.5             2          0.92            0.75                  0.0

ethiopia_highlands
      s1_median_per_month  s1_min_month  dw_cov1_mean  d

In [7]:
# 2b. Make the 4 sites 'train' in regions.yaml, then retrain V12 (2 seeds) in a fresh folder
import re
P = 'ethiopia_highlands cerrado_brazil kazakh_steppe texas_rangeland'.split()
s = open('configs/regions.yaml').read()
for k in P: s = re.sub(r'(^\s*'+k+r':\s*\{[^}]*?split:\s*)test_ood', r'\1train', s, flags=re.M)
open('configs/regions.yaml','w').write(s); print('promoted', P)
RUNS_grass = f'{BASE}/runs_grass'; copy_baselines(RUNS_grass)
%run scripts/train_all.py --data $OUT --out $RUNS_grass --variants V12_window3_balanced_stats --seeds 0 1 --workers 0

promoted ['ethiopia_highlands', 'cerrado_brazil', 'kazakh_steppe', 'texas_rangeland']
V12_window3_balanced_stats: encoder weights = imagenet (V0 0.466 vs V1 0.597)
loaded 25 train and 6 val regions
=== V12_window3_balanced_stats seed 0
train patches 11803, val patches 600, class freq [0.171, 0.09, 0.178, 0.46, 0.058, 0.043]
ep 1/12 loss 1.418 val mIoU 0.601 OA 0.797 (17s)
ep 2/12 loss 1.064 val mIoU 0.615 OA 0.810 (20s)
ep 3/12 loss 0.935 val mIoU 0.610 OA 0.807 (16s)
ep 4/12 loss 0.882 val mIoU 0.482 OA 0.773 (17s)
ep 5/12 loss 0.817 val mIoU 0.542 OA 0.795 (16s)
ep 6/12 loss 0.778 val mIoU 0.624 OA 0.818 (17s)
ep 7/12 loss 0.738 val mIoU 0.521 OA 0.790 (16s)
ep 8/12 loss 0.703 val mIoU 0.510 OA 0.786 (15s)
ep 9/12 loss 0.683 val mIoU 0.490 OA 0.777 (17s)
ep 10/12 loss 0.700 val mIoU 0.476 OA 0.771 (17s)
ep 11/12 loss 0.687 val mIoU 0.458 OA 0.764 (17s)
early stop
=== V12_window3_balanced_stats seed 1
train patches 11803, val patches 600, class freq [0.171, 0.09, 0.178, 0.46, 0.058, 0

In [8]:
# 2c. Evaluate and compare with the shipped V12 on the SAME regions
%run scripts/evaluate.py --data $OUT --runs $RUNS_grass --variant V12_window3_balanced_stats --tag _grass
# shipped V12 on the same config (for a fair side-by-side):
%run scripts/evaluate.py --data $OUT --runs $RUNS --variant V12_window3_balanced_stats --tag _grass_ref
print('Compare dw_rural, Grass/shrub IoU and Water IoU between the two. Keep only if Grass/shrub rises and Water holds.')

2 seeds, window 3, coherence False, yearly stats True, year 2021
Esri scheme: original
prague       mIoU vs DW 0.561  OA 0.876
frankfurt    mIoU vs DW 0.539  OA 0.867
milan        mIoU vs DW 0.506  OA 0.881
valencia     mIoU vs DW 0.457  OA 0.795
north_china_plain mIoU vs DW 0.380  OA 0.873
appalachians mIoU vs DW 0.138  OA 0.674
western_desert mIoU vs DW 1.000  OA 1.000
great_basin  mIoU vs DW 0.051  OA 0.209
lake_balaton mIoU vs DW 0.558  OA 0.950
za_freestate_maize mIoU vs DW 0.393  OA 0.732
za_lowveld_savanna mIoU vs DW 0.238  OA 0.450
za_karoo_shrub mIoU vs DW 0.186  OA 0.660
za_capetown  mIoU vs DW 0.325  OA 0.450
india_punjab mIoU vs DW 0.290  OA 0.789
deccan_india mIoU vs DW 0.344  OA 0.587
java_rice    mIoU vs DW 0.162  OA 0.617
mekong_delta mIoU vs DW 0.351  OA 0.710
amazon_para  mIoU vs DW 0.436  OA 0.911
australia_outback mIoU vs DW 0.175  OA 0.694
altiplano_andes mIoU vs DW 0.319  OA 0.885
sahel_niger  mIoU vs DW 0.024  OA 0.033
iowa_maize   mIoU vs DW 0.429  OA 0.895
cong

## 3. Bigger backbone: ResNet-50 (one retrain)
More capacity may hold cities and rural together. Trains V14 (resnet50) on the current training set. Heavier per epoch.

In [9]:
# 3. Train V14 (ResNet-50), 2 seeds, then evaluate
RUNS_r50 = f'{BASE}/runs_r50'; copy_baselines(RUNS_r50)
%run scripts/train_all.py --data $OUT --out $RUNS_r50 --variants V14_resnet50 --seeds 0 1 --workers 0
%run scripts/evaluate.py --data $OUT --runs $RUNS_r50 --variant V14_resnet50 --tag _r50

V14_resnet50: encoder weights = imagenet (V0 0.466 vs V1 0.597)
loaded 25 train and 6 val regions
=== V14_resnet50 seed 0
train patches 11803, val patches 600, class freq [0.171, 0.09, 0.178, 0.46, 0.058, 0.043]


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ep 1/12 loss 1.355 val mIoU 0.585 OA 0.794 (21s)
ep 2/12 loss 1.022 val mIoU 0.575 OA 0.802 (22s)
ep 3/12 loss 0.897 val mIoU 0.595 OA 0.810 (20s)
ep 4/12 loss 0.842 val mIoU 0.499 OA 0.781 (23s)
ep 5/12 loss 0.789 val mIoU 0.505 OA 0.785 (20s)
ep 6/12 loss 0.747 val mIoU 0.517 OA 0.790 (21s)
ep 7/12 loss 0.713 val mIoU 0.500 OA 0.783 (22s)
ep 8/12 loss 0.690 val mIoU 0.514 OA 0.790 (20s)
early stop
=== V14_resnet50 seed 1
train patches 11803, val patches 600, class freq [0.171, 0.09, 0.178, 0.46, 0.058, 0.043]
ep 1/12 loss 1.397 val mIoU 0.611 OA 0.807 (22s)
ep 2/12 loss 0.991 val mIoU 0.622 OA 0.814 (22s)
ep 3/12 loss 0.893 val mIoU 0.624 OA 0.815 (20s)
ep 4/12 loss 0.864 val mIoU 0.645 OA 0.822 (21s)
ep 5/12 loss 0.782 val mIoU 0.643 OA 0.822 (20s)
ep 6/12 loss 0.780 val mIoU 0.647 OA 0.826 (21s)
ep 7/12 loss 0.710 val mIoU 0.651 OA 0.827 (20s)
ep 8/12 loss 0.699 val mIoU 0.652 OA 0.828 (18s)
ep 9/12 loss 0.694 val mIoU 0.652 OA 0.829 (20s)
ep 10/12 loss 0.661 val mIoU 0.651 OA 0.83

## 4. Train on WorldCover labels instead of Dynamic World (one retrain)
Better labels in some rural areas, but WorldCover then stops being an independent check, so this model is scored against
Esri only. Needs reference.npz for every TRAIN region first.

In [5]:
# 4a. Export the WorldCover/Esri reference for the TRAIN regions (fast; ~10 s per region)
TRAIN_REGS = 'munich berlin hamburg cologne paris amsterdam warsaw rome madrid vienna poltava hungarian_plain black_forest sahara_algeria dehesa_spain finnish_lakeland pampas nsw_pasture kenya_savanna mongolia_steppe arabia_desert'
%run scripts/export_reference.py --project $PROJECT --out $OUT --regions $TRAIN_REGS

munich: exists, skipped
berlin: exists, skipped
hamburg: exists, skipped
cologne: exists, skipped
paris: exists, skipped
amsterdam: exists, skipped
warsaw: exists, skipped
rome: exists, skipped
madrid: exists, skipped
vienna: exists, skipped
poltava: exists, skipped
hungarian_plain: exists, skipped
black_forest: exists, skipped
sahara_algeria: exists, skipped
dehesa_spain: exists, skipped
finnish_lakeland: exists, skipped
pampas: exists, skipped
nsw_pasture: exists, skipped
kenya_savanna: exists, skipped
mongolia_steppe: exists, skipped
arabia_desert: exists, skipped


In [6]:
# 4b. Train V15 (WorldCover labels), 2 seeds, then evaluate (read the Esri rows, not WorldCover)
RUNS_wc = f'{BASE}/runs_wc'; copy_baselines(RUNS_wc)
%run scripts/train_all.py --data $OUT --out $RUNS_wc --variants V15_worldcover_labels --seeds 0 1 --workers 0
%run scripts/evaluate.py --data $OUT --runs $RUNS_wc --variant V15_worldcover_labels --tag _wc
print('WorldCover is now part of training: compare the ESRI rows (esri, esri_rural) with the shipped V12, not the wc rows.')

V15_worldcover_labels: encoder weights = imagenet (V0 0.466 vs V1 0.597)
loaded 21 train and 6 val regions
skip /home/jovyan/vault/lulc/runs_wc/V15_worldcover_labels/seed0
skip /home/jovyan/vault/lulc/runs_wc/V15_worldcover_labels/seed1

Validation mIoU (gaps under 0.05 are inside seed noise):
                         mean    std  count
variant                                   
V0_scratch_1m          0.466  0.010      3
V15_worldcover_labels  0.559  0.010      2
V1_imagenet_1m         0.597  0.008      3
2 seeds, window 3, coherence False, yearly stats True, year 2021
Esri scheme: original
prague       mIoU vs DW 0.434  OA 0.713
frankfurt    mIoU vs DW 0.447  OA 0.766
milan        mIoU vs DW 0.457  OA 0.827
valencia     mIoU vs DW 0.265  OA 0.483
north_china_plain mIoU vs DW 0.244  OA 0.763
appalachians mIoU vs DW 0.140  OA 0.775
western_desert mIoU vs DW 1.000  OA 1.000
great_basin  mIoU vs DW 0.098  OA 0.508
lake_balaton mIoU vs DW 0.389  OA 0.827
za_freestate_maize mIoU vs DW 0.243

## Decision
Keep the shipped **V12 city-trained** unless one of V14 / V15 / the grass retrain clearly beats it on the same held-out
regions (all mIoU and cities not worse, rural or Grass/shrub up). If none does, that is itself a reportable result:
the model is label- and signal-limited, not architecture- or data-limited.